# Fast MVTec AD preprocessing audit

Run this separately from `01_dataset_exploration.ipynb`. It does **not** modify or replace that notebook or its saved outputs.

Purpose: collect enough evidence to design preprocessing. It checks file structure and image/mask metadata across the full dataset, but reads pixel values only from small, deterministic samples to avoid slow Google Drive I/O.

Outputs are saved in `CS663_Shared/reports/preprocessing_audit_fast/`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import csv, json, random
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter, defaultdict

DATA_ROOT = Path('/content/drive/MyDrive/CS663_Shared/datasets/mvtec_anomaly_detection')
OUT_DIR = Path('/content/drive/MyDrive/CS663_Shared/reports/preprocessing_audit_fast')
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
PIXEL_SAMPLES_PER_GROUP = 5
MASK_SAMPLES_PER_DEFECT = 5

assert DATA_ROOT.is_dir(), f'Dataset root not found: {DATA_ROOT}'
categories = sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir() and (p/'train'/'good').is_dir())
print(f'Found {len(categories)} categories:', categories)


Mounted at /content/drive
Found 15 categories: ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


## 1. Full structural and image metadata scan
Opens every image only far enough to read its header (dimensions, mode, format). It does not decode all pixel arrays.

In [2]:
image_rows = []
mask_rows = []
missing_masks = []
def read_header(path):
    with Image.open(path) as im:
        return {'width': im.width, 'height': im.height, 'mode': im.mode,
                'format': im.format, 'bands': str(im.getbands()),
                'aspect_ratio': round(im.width / im.height, 6) if im.height else None}

for category in categories:
    base = DATA_ROOT / category
    for split_dir in [base/'train'/'good', base/'test']:
        if not split_dir.exists():
            continue
        split = 'train' if split_dir.parent.name == 'train' else 'test'
        for p in sorted(split_dir.rglob('*.png')):
            defect = 'good' if split == 'train' else p.parent.name
            try:
                meta = read_header(p)
                image_rows.append({'category':category, 'split':split, 'defect_type':defect,
                                   'path':str(p), **meta})
            except Exception as e:
                image_rows.append({'category':category, 'split':split, 'defect_type':defect,
                                   'path':str(p), 'read_error':repr(e)})
            if split == 'test' and defect != 'good':
                mp = base/'ground_truth'/defect/f'{p.stem}_mask.png'
                if not mp.exists():
                    missing_masks.append({'category':category,'defect_type':defect,
                                          'image_path':str(p),'expected_mask_path':str(mp)})

    gt = base/'ground_truth'
    if gt.exists():
        for mp in sorted(gt.rglob('*.png')):
            try:
                meta = read_header(mp)
                mask_rows.append({'category':category,'defect_type':mp.parent.name,
                                  'path':str(mp),**meta})
            except Exception as e:
                mask_rows.append({'category':category,'defect_type':mp.parent.name,
                                  'path':str(mp),'read_error':repr(e)})

images_df = pd.DataFrame(image_rows)
masks_df = pd.DataFrame(mask_rows)
missing_df = pd.DataFrame(missing_masks)
images_df.to_csv(OUT_DIR/'all_image_metadata.csv', index=False)
masks_df.to_csv(OUT_DIR/'all_mask_metadata.csv', index=False)
missing_df.to_csv(OUT_DIR/'missing_mask_pairs.csv', index=False)
print('Images scanned:', len(images_df), '| masks scanned:', len(masks_df), '| missing mask pairs:', len(missing_masks))
display(images_df.groupby(['split','mode','width','height'], dropna=False).size().rename('count').reset_index().sort_values('count', ascending=False).head(30))
display(masks_df.groupby(['mode','width','height'], dropna=False).size().rename('count').reset_index().sort_values('count', ascending=False).head(30))


Images scanned: 5354 | masks scanned: 1258 | missing mask pairs: 0


,split,mode,width,height,count
13,train,RGB,1024,1024,1660
7,train,L,1024,1024,824
6,test,RGB,1024,1024,722
0,test,L,1024,1024,389
9,train,RGB,800,800,267
10,train,RGB,840,840,230
8,train,RGB,700,700,220
12,train,RGB,1000,1000,219
11,train,RGB,900,900,209
2,test,RGB,800,800,167


,mode,width,height,count
5,L,1024,1024,768
1,L,800,800,141
4,L,1000,1000,109
0,L,700,700,93
2,L,840,840,84
3,L,900,900,63


## 2. Pixel statistics from deterministic samples
Reads only up to five images per category and split. This is for colour-channel, dtype, range, and intensity evidence—not an exhaustive pixel census.

In [3]:
rng = random.Random(SEED)
pixel_rows = []
for category in categories:
    base = DATA_ROOT/category
    groups = {'train_good': sorted((base/'train'/'good').glob('*.png'))}
    test_root = base/'test'
    if test_root.exists():
        for defect_dir in sorted(p for p in test_root.iterdir() if p.is_dir()):
            groups[f'test_{defect_dir.name}'] = sorted(defect_dir.glob('*.png'))
    for group, paths in groups.items():
        if not paths: continue
        chosen = rng.sample(paths, min(PIXEL_SAMPLES_PER_GROUP, len(paths)))
        for p in chosen:
            try:
                with Image.open(p) as im:
                    arr = np.asarray(im)
                row = {'category':category,'group':group,'file':p.name,
                       'shape':str(arr.shape),'dtype':str(arr.dtype),
                       'min':float(np.min(arr)),'max':float(np.max(arr)),
                       'mean':float(np.mean(arr)),'std':float(np.std(arr))}
                if arr.ndim == 3:
                    for c in range(arr.shape[2]):
                        row[f'ch{c}_mean'] = float(np.mean(arr[:,:,c]))
                        row[f'ch{c}_std'] = float(np.std(arr[:,:,c]))
                pixel_rows.append(row)
            except Exception as e:
                pixel_rows.append({'category':category,'group':group,'file':p.name,'read_error':repr(e)})

pixels_df = pd.DataFrame(pixel_rows)
pixels_df.to_csv(OUT_DIR/'sampled_image_pixel_statistics.csv', index=False)
display(pixels_df.head(20))
print('Pixel arrays decoded:', len(pixels_df))


,category,group,file,shape,dtype,min,max,mean,std,ch0_mean,ch0_std,ch1_mean,ch1_std,ch2_mean,ch2_std
0,bottle,train_good,163.png,"(900, 900, 3)",uint8,25.0,255.0,139.927894,91.412846,139.703494,92.647101,138.472478,92.613466,141.607710,88.899920
1,bottle,train_good,028.png,"(900, 900, 3)",uint8,25.0,255.0,137.189982,92.356320,136.652462,93.791403,135.340578,93.797179,139.576906,89.358407
2,bottle,train_good,006.png,"(900, 900, 3)",uint8,27.0,255.0,138.141384,91.044917,137.543186,92.380646,136.526023,92.298129,140.354941,88.354598
3,bottle,train_good,189.png,"(900, 900, 3)",uint8,27.0,255.0,138.744553,91.444186,137.997075,92.836174,137.172648,92.777961,141.063935,88.608145
4,bottle,train_good,070.png,"(900, 900, 3)",uint8,27.0,255.0,139.142956,91.836818,138.272457,93.276417,137.517981,93.217724,141.638430,88.895126
5,bottle,test_broken_large,007.png,"(900, 900, 3)",uint8,27.0,255.0,138.963075,91.791755,138.080407,93.267382,137.158056,93.286383,141.650763,88.683506
6,bottle,test_broken_large,019.png,"(900, 900, 3)",uint8,27.0,255.0,139.474066,90.807173,138.935909,92.017434,137.800728,92.192600,141.685562,88.108821
7,bottle,test_broken_large,004.png,"(900, 900, 3)",uint8,27.0,255.0,138.992403,91.084809,138.404793,92.567669,137.185340,92.566426,141.387077,87.992416
8,bottle,test_broken_large,003.png,"(900, 900, 3)",uint8,27.0,255.0,134.391531,93.904088,133.045820,95.571203,132.526227,95.500273,137.602547,90.466431
9,bottle,test_broken_large,002.png,"(900, 900, 3)",uint8,29.0,255.0,136.881942,91.297588,135.581714,92.791751,135.093470,92.787043,139.970641,88.156538


Pixel arrays decoded: 515


## 3. Ground-truth mask value and coverage samples
All mask headers were scanned above. Here, decode only up to five masks per category/defect type to inspect unique values and foreground coverage.

In [4]:
mask_pixel_rows = []
if len(masks_df):
    for (category, defect), group in masks_df.groupby(['category','defect_type']):
        paths = sorted(Path(p) for p in group['path'].tolist())
        chosen = rng.sample(paths, min(MASK_SAMPLES_PER_DEFECT, len(paths)))
        for p in chosen:
            try:
                with Image.open(p) as im:
                    arr = np.asarray(im)
                vals, counts = np.unique(arr, return_counts=True)
                # MVTec masks conventionally use zero for background; report raw values and nonzero fraction.
                mask_pixel_rows.append({'category':category,'defect_type':defect,'file':p.name,
                    'shape':str(arr.shape),'dtype':str(arr.dtype),
                    'unique_values':json.dumps(vals.tolist()),
                    'value_counts':json.dumps(counts.tolist()),
                    'nonzero_fraction':float(np.count_nonzero(arr)/arr.size)})
            except Exception as e:
                mask_pixel_rows.append({'category':category,'defect_type':defect,'file':p.name,'read_error':repr(e)})
mask_pixels_df = pd.DataFrame(mask_pixel_rows)
mask_pixels_df.to_csv(OUT_DIR/'sampled_mask_pixel_audit.csv', index=False)
display(mask_pixels_df.head(30))
print('Mask pixel arrays decoded:', len(mask_pixels_df))


,category,defect_type,file,shape,dtype,unique_values,value_counts,nonzero_fraction
0,bottle,broken_large,009_mask.png,"(900, 900)",uint8,"[0, 255]","[739125, 70875]",0.087500
1,bottle,broken_large,016_mask.png,"(900, 900)",uint8,"[0, 255]","[691087, 118913]",0.146806
2,bottle,broken_large,019_mask.png,"(900, 900)",uint8,"[0, 255]","[707783, 102217]",0.126194
3,bottle,broken_large,013_mask.png,"(900, 900)",uint8,"[0, 255]","[700710, 109290]",0.134926
4,bottle,broken_large,010_mask.png,"(900, 900)",uint8,"[0, 255]","[719972, 90028]",0.111146
5,bottle,broken_small,012_mask.png,"(900, 900)",uint8,"[0, 255]","[746809, 63191]",0.078014
6,bottle,broken_small,009_mask.png,"(900, 900)",uint8,"[0, 255]","[790904, 19096]",0.023575
7,bottle,broken_small,017_mask.png,"(900, 900)",uint8,"[0, 255]","[789616, 20384]",0.025165
8,bottle,broken_small,004_mask.png,"(900, 900)",uint8,"[0, 255]","[784074, 25926]",0.032007
9,bottle,broken_small,006_mask.png,"(900, 900)",uint8,"[0, 255]","[788203, 21797]",0.026910


Mask pixel arrays decoded: 365


## 4. Compact preprocessing handoff for Claude
This writes a concise JSON summary from the full header scans and sampled pixel audits. Inspect the CSVs for details.

In [5]:
def frequency(series):
    return {str(k):int(v) for k,v in series.value_counts(dropna=False).items()}

handoff = {
  'dataset_root_layout': 'root/category/{train/good, test/defect_type, ground_truth/defect_type}',
  'categories': categories,
  'full_scan_counts': {'images':int(len(images_df)), 'masks':int(len(masks_df)),
                       'missing_anomalous_test_masks':int(len(missing_masks))},
  'image_modes':frequency(images_df['mode']) if 'mode' in images_df else {},
  'image_formats':frequency(images_df['format']) if 'format' in images_df else {},
  'image_dimensions':frequency(images_df['width'].astype(str)+'x'+images_df['height'].astype(str)) if {'width','height'}.issubset(images_df.columns) else {},
  'mask_modes':frequency(masks_df['mode']) if 'mode' in masks_df else {},
  'mask_formats':frequency(masks_df['format']) if 'format' in masks_df else {},
  'mask_dimensions':frequency(masks_df['width'].astype(str)+'x'+masks_df['height'].astype(str)) if {'width','height'}.issubset(masks_df.columns) else {},
  'pixel_sample_count':int(len(pixels_df)),
  'mask_pixel_sample_count':int(len(mask_pixels_df)),
  'pixel_samples_csv':'sampled_image_pixel_statistics.csv',
  'mask_samples_csv':'sampled_mask_pixel_audit.csv',
  'full_image_metadata_csv':'all_image_metadata.csv',
  'full_mask_metadata_csv':'all_mask_metadata.csv',
  'missing_mask_pairs_csv':'missing_mask_pairs.csv',
  'loader_contract': 'Dataset loader returns original NumPy arrays; no resizing, colour conversion, scaling, normalization, thresholding, or tensor conversion.',
  'interpretation_note': 'Header metadata is scanned across all files. Pixel statistics and mask value/coverage are sampled deterministically; do not treat sample statistics as exhaustive.'
}
with open(OUT_DIR/'preprocessing_handoff_fast.json','w') as f:
    json.dump(handoff,f,indent=2)
print(json.dumps(handoff,indent=2))
print('\nSaved all audit outputs to:', OUT_DIR)


{
  "dataset_root_layout": "root/category/{train/good, test/defect_type, ground_truth/defect_type}",
  "categories": [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper"
  ],
  "full_scan_counts": {
    "images": 5354,
    "masks": 1258,
    "missing_anomalous_test_masks": 0
  },
  "image_modes": {
    "RGB": 4141,
    "L": 1213
  },
  "image_formats": {
    "PNG": 5354
  },
  "image_dimensions": {
    "1024x1024": 3595,
    "800x800": 434,
    "1000x1000": 351,
    "840x840": 347,
    "700x700": 335,
    "900x900": 292
  },
  "mask_modes": {
    "L": 1258
  },
  "mask_formats": {
    "PNG": 1258
  },
  "mask_dimensions": {
    "1024x1024": 768,
    "800x800": 141,
    "1000x1000": 109,
    "700x700": 93,
    "840x840": 84,
    "900x900": 63
  },
  "pixel_sample_count": 515,
  "mask_pixel_sample_count": 365,
  "pixel_samples_